# Validate PLUMED conda packages on an NVIDIA GPU
Select a GPU runtime first. Upload the **Linux x86-64 conda** and **OpenACC** artifact ZIPs from the same final CI run. This notebook tests installed packages; it does not rebuild plugins or override CUDA driver compatibility. All three variants must pass to validate the complete GPU matrix. Keep the logs even when a solve fails because the driver is too old.


In [ ]:
import subprocess
subprocess.run(["nvidia-smi"], check=True)


In [ ]:
%%bash
set -euo pipefail
curl -fLsS https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -o /content/miniforge.sh
bash /content/miniforge.sh -b -p /content/plumed-conda
/content/plumed-conda/bin/conda install -y -n base -c conda-forge conda-build
# Use the exact implementation commit corresponding to the selected CI artifacts.
git clone --filter=blob:none --single-branch --branch agents/conda-packaging/extensions https://github.com/GiovanniBussi/plumed2.git /content/plumed-source


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, shutil
uploaded = files.upload()
channel = Path("/content/plumed-channel/linux-64")
channel.mkdir(parents=True, exist_ok=True)
for filename in uploaded:
    with zipfile.ZipFile(filename) as archive:
        for member in archive.infolist():
            if member.filename.endswith(".conda"):
                with archive.open(member) as source, (channel / Path(member.filename).name).open("wb") as target:
                    shutil.copyfileobj(source, target)
assert list(channel.glob("plumed-ext-cuda-coordination-*.conda")), "Missing CUDA packages"
assert list(channel.glob("plumed-ext-openacc-ptm-*.conda")), "Missing OpenACC package"


In [ ]:
%%bash
set -euo pipefail
export PATH=/content/plumed-conda/bin:$PATH
conda index /content/plumed-channel
export VERSION=2.11.0dev4
export GPU_CONDA_CHANNEL=file:///content/plumed-channel
export GPU_TEST_ROOT=/content/plumed-gpu-results
bash /content/plumed-source/conda/run-gpu-test.sh


In [ ]:
from google.colab import files
import shutil
shutil.make_archive("/content/plumed-gpu-results", "zip", "/content/plumed-gpu-results")
files.download("/content/plumed-gpu-results.zip")
